# 02. 전처리 파이프라인

모델 학습 전에 원시 리뷰 데이터를 세 단계로 가공합니다.

| 단계 | 목적 |
|------|------|
| **k-core 필터링** | 리뷰가 너무 적은 유저·아이템 제거 (cold-start 완화) |
| **ID 인코딩** | 문자열 ID → 정수 (Embedding 레이어 입력 형식) |
| **Leave-one-out split** | 평가 표준 방식으로 train/test 분리 |

In [ ]:
import sys
sys.path.append('..')

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

DATA_DIR = Path('../data/processed/All_Beauty')

## 1. 원시 데이터 탐색 (DuckDB)

DuckDB는 parquet 파일을 메모리에 전부 올리지 않고 직접 쿼리합니다.
집계·필터링이 pandas보다 빠르고, SQL로 작성할 수 있습니다.

In [ ]:
reviews_path = str(DATA_DIR / 'reviews.parquet')

# 전체 통계 — DuckDB가 parquet을 직접 쿼리
duckdb.sql(f"""
    SELECT
        COUNT(*)                    AS total_interactions,
        COUNT(DISTINCT user_id)     AS n_users,
        COUNT(DISTINCT parent_asin) AS n_items,
        ROUND(AVG(rating), 2)       AS avg_rating,
        MIN(timestamp)              AS ts_min,
        MAX(timestamp)              AS ts_max
    FROM '{reviews_path}'
""")

In [ ]:
# 유저별 리뷰 수 분포 — 추천 시스템에서 핵심 지표
user_review_counts = duckdb.sql(f"""
    SELECT user_id, COUNT(*) AS n_reviews
    FROM '{reviews_path}'
    GROUP BY user_id
""").df()

print("유저별 리뷰 수 분포:")
print(user_review_counts['n_reviews'].describe().round(2))
print()

# k별 생존 유저 수
print("k값별 생존 유저 비율:")
for k in [1, 2, 3, 5, 10]:
    cnt = (user_review_counts['n_reviews'] >= k).sum()
    pct = cnt / len(user_review_counts) * 100
    print(f"  k>={k:2d} : {cnt:>7,}명 ({pct:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 유저별 리뷰 수 (롱테일 확인)
user_review_counts['n_reviews'].clip(upper=20).value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue'
)
axes[0].set_title('유저별 리뷰 수 분포 (capped 20)')
axes[0].set_xlabel('리뷰 수')
axes[0].set_ylabel('유저 수')

# 아이템별 리뷰 수
item_review_counts = duckdb.sql(f"""
    SELECT parent_asin, COUNT(*) AS n_reviews
    FROM '{reviews_path}'
    GROUP BY parent_asin
""").df()
item_review_counts['n_reviews'].clip(upper=30).value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral'
)
axes[1].set_title('아이템별 리뷰 수 분포 (capped 30)')
axes[1].set_xlabel('리뷰 수')
axes[1].set_ylabel('아이템 수')

plt.tight_layout()
plt.show()

print(f"\n리뷰 1개뿐인 유저: {(user_review_counts['n_reviews'] == 1).sum():,}명 "
      f"({(user_review_counts['n_reviews'] == 1).mean():.1%}) ← 이들이 cold-start 문제")

## 2. k-core 필터링

### 왜 필요한가?
리뷰 1~2개뿐인 유저는 선호 패턴을 학습하기 어렵습니다.  
아이템도 마찬가지 — 리뷰가 없으면 임베딩이 제대로 학습되지 않습니다.

### 작동 방식
```
while 변화가 있는 동안:
    리뷰 수 < k인 유저 제거
    리뷰 수 < k인 아이템 제거
```
수렴까지 반복하는 이유: 유저를 제거하면 일부 아이템 리뷰 수가 k 미만이 될 수 있고, 반대도 마찬가지입니다.

### k값 선택: All_Beauty의 경우
- 유저의 92.3%가 리뷰 1개 → k=5면 데이터 99.5% 손실
- **k=3** 선택: 12K interactions / 2K users — SASRec 평균 시퀀스 길이 ~5.9

In [ ]:
# k별 데이터 크기 비교
def kcore_size(df, k):
    while True:
        n = len(df)
        df = df[df['user_id'].isin(df.groupby('user_id').size()[lambda x: x >= k].index)]
        df = df[df['parent_asin'].isin(df.groupby('parent_asin').size()[lambda x: x >= k].index)]
        if len(df) == n:
            break
    return df

raw_df = duckdb.sql(f"SELECT user_id, parent_asin FROM '{reviews_path}'").df()

rows = []
for k in [2, 3, 5]:
    filtered = kcore_size(raw_df.copy(), k)
    u = filtered['user_id'].nunique()
    i = filtered['parent_asin'].nunique()
    sp = 1 - len(filtered) / (u * i)
    rows.append({'k': k, 'interactions': len(filtered), 'users': u, 'items': i,
                 'sparsity': f'{sp:.4%}',
                 'avg_seq_len': round(len(filtered) / u, 1)})

pd.DataFrame(rows).set_index('k')

## 3. 전처리 파이프라인 실행

`scripts/preprocess.py`를 실행합니다.  
k-core(k=3) → ID 인코딩 → leave-one-out split → parquet 저장

In [ ]:
from scripts.preprocess import run
meta = run(k=3)

## 4. 결과 검증

### ID 인코딩 확인
원래 문자열 ID가 정수로 매핑됐는지 확인합니다.  
**왜 정수가 필요한가?** PyTorch의 `nn.Embedding(n_users, dim)`은  
인덱스 0 ~ n_users-1 범위의 정수만 입력받습니다.

In [ ]:
train = pd.read_parquet(DATA_DIR / 'train.parquet')
test  = pd.read_parquet(DATA_DIR / 'test.parquet')

print("train 샘플:")
display(train.head(5))

print("\nID 범위 검증:")
print(f"  user_id: {train['user_id'].min()} ~ {train['user_id'].max()} (기대: 0 ~ {meta['n_users']-1})")
print(f"  item_id: {train['parent_asin'].min()} ~ {train['parent_asin'].max()} (기대: 0 ~ {meta['n_items']-1})")

### Leave-one-out split 확인
각 유저의 마지막 상호작용이 test에 있는지 확인합니다.

In [ ]:
# 유저 0의 전체 이력 확인
all_data = pd.concat([train, test]).sort_values(['user_id', 'timestamp'])
user0 = all_data[all_data['user_id'] == 0]

print("유저 0의 전체 이력 (timestamp 순):")
display(user0[['user_id', 'parent_asin', 'rating', 'timestamp']])

test_item = test[test['user_id'] == 0]['parent_asin'].values[0]
last_item = user0.iloc[-1]['parent_asin']
print(f"\ntest 아이템: {test_item}  |  이력의 마지막 아이템: {last_item}")
print(f"일치 여부: {test_item == last_item} ✓" if test_item == last_item else f"불일치 ✗")

### 최종 데이터셋 요약

In [ ]:
with open(DATA_DIR / 'dataset_meta.json') as f:
    meta = json.load(f)

summary = pd.DataFrame([{
    'n_users':     meta['n_users'],
    'n_items':     meta['n_items'],
    'n_train':     meta['n_train'],
    'n_test':      meta['n_test'],
    'sparsity':    f"{meta['sparsity']:.4%}",
    'avg_seq_len': round(meta['n_train'] / meta['n_users'], 1),
}], index=['All_Beauty (k=3)'])
display(summary)

print()
print("생성된 파일:")
for f in sorted(DATA_DIR.iterdir()):
    print(f"  {f.name}")

---
## 요약 — 이 전처리가 모델에 미치는 영향

| 전처리 | 모델 관점 |
|--------|----------|
| k-core 필터링 | 학습 데이터 품질 ↑, 전체 양 ↓ — cold-start는 별도 전략 필요 |
| ID 인코딩 | `nn.Embedding` 레이어의 입력 형식 맞춤 |
| Leave-one-out | 모든 유저에 대해 동일 조건으로 평가 가능 (Hit@K, NDCG@K) |

다음 노트북: **03_mf_baseline.ipynb** — 이 train/test를 사용해 Matrix Factorization 구현